# ViT-Small: ensamble multi-vista, umbral clínico y validación

Evalúa el ensamble multi-vista de ViT-Small (22 M, multiseed) en MRNet. Carga los tres planos,
promedia las probabilidades, fija el umbral de decisión sobre validación (recall máximo con
precisión >= 0,75) y lo aplica al conjunto de test. Es el equivalente, para ViT-Small, de los
notebooks de umbral de `experiments/cnn` y `pipeline`.

Requiere los pesos `checkpoints/vit_small_multiseed/best_{plano}_multiseed_final.pth` y los datos de
MRNet (ver `MODELS.md` y `DATA.md`).

In [1]:
import os
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
import sys
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

BASE = Path('/home/palodo2/acl_classifier')
sys.path.insert(0, str(BASE))
from src.data_loader import OptimizedMRNetDataset, get_val_test_transform
from src.models import ViTSmallMultiSliceClassifier

DEVICE = torch.device('cpu')
DATA = BASE / 'data'
CKPT_DIR = BASE / 'checkpoints' / 'vit_small_multiseed'
PLANES = ['sagittal', 'coronal', 'axial']
# Pooling por plano (igual que en el entrenamiento)
POOLING = {'sagittal': 'attention', 'coronal': 'max', 'axial': 'attention'}
print('Dispositivo:', DEVICE)

/home/palodo2/acl_classifier/.venv/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:11: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.sparse import csr_matrix, issparse


Dispositivo: cpu


In [2]:
def calc(labels, probs, thr):
    pred = (probs >= thr).astype(int)
    tp = int(((pred == 1) & (labels == 1)).sum()); tn = int(((pred == 0) & (labels == 0)).sum())
    fp = int(((pred == 1) & (labels == 0)).sum()); fn = int(((pred == 0) & (labels == 1)).sum())
    acc = (tp + tn) / (tp + tn + fp + fn)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, acc=acc, prec=prec, rec=rec, spec=spec, f1=f1)

def find_threshold(labels, probs, min_precision=0.75):
    best_t, best_r = 0.5, -1.0
    for t in np.linspace(0, 1, 1000):
        m = calc(labels, probs, t)
        if m['prec'] >= min_precision and m['rec'] > best_r:
            best_r, best_t = m['rec'], t
    return best_t

In [3]:
def plane_probs(plane, split):
    csv = DATA / ('val-acl.csv' if split == 'val' else 'test-acl.csv')
    root = DATA / ('val' if split == 'val' else 'test')
    ds = OptimizedMRNetDataset(csv_path=str(csv), data_root=str(root), plane=plane,
        indices_cache_path=str(DATA / 'slice_indices_final' / f'{split}_{plane}_indices.json'),
        transform=get_val_test_transform())
    loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=0)
    # pretrained=True para construir la config correcta de ViT-Small (dim 384)
    model = ViTSmallMultiSliceClassifier(pretrained=True, pooling_mode=POOLING[plane]).to(DEVICE)
    ck = torch.load(CKPT_DIR / f'best_{plane}_multiseed_final.pth', map_location='cpu', weights_only=False)
    sd = ck.get('model_state_dict', ck)
    sd = {k.replace('module.', '', 1) if k.startswith('module.') else k: v for k, v in sd.items()}
    model.load_state_dict(sd, strict=False)
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(batch[0].to(DEVICE))
            logits = out[0] if isinstance(out, tuple) else out
            probs.extend(torch.sigmoid(logits).cpu().numpy().flatten()); labels.extend(batch[1].numpy())
    return np.array(probs), np.array(labels)

def ensemble(split):
    pp, labels = {}, None
    for p in PLANES:
        pr, lb = plane_probs(p, split); pp[p] = pr; labels = lb if labels is None else labels
        print(f'  AUC {p:8s} ({split}): {roc_auc_score(lb, pr):.4f}')
    ens = (pp['sagittal'] + pp['coronal'] + pp['axial']) / 3.0
    return ens, labels

## Validación

Ensamble en validación y búsqueda del umbral clínico.

In [4]:
val_ens, val_lb = ensemble('val')
thr = find_threshold(val_lb, val_ens, 0.75)
mv = calc(val_lb, val_ens, thr)
print(f'\nUmbral tau* = {thr:.4f}  (n={len(val_lb)})')
print(f"AUC {roc_auc_score(val_lb, val_ens):.4f}  Acc {mv['acc']:.4f}  Prec {mv['prec']:.4f}  "
      f"Recall {mv['rec']:.4f}  Esp {mv['spec']:.4f}  F1 {mv['f1']:.4f}")
print(f"VP {mv['tp']}  FN {mv['fn']}  FP {mv['fp']}  VN {mv['tn']}")

Some weights of ViTModel were not initialized from the model checkpoint at WinKawaks/vit-small-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Dataset cargado: 188 casos
  Plane: sagittal
  Augmentation: None (usando transform directo)
  Índices cacheados: /home/palodo2/acl_classifier/data/slice_indices_final/val_sagittal_indices.json


Some weights of ViTModel were not initialized from the model checkpoint at WinKawaks/vit-small-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  AUC sagittal (val): 0.9764
 Dataset cargado: 188 casos
  Plane: coronal
  Augmentation: None (usando transform directo)
  Índices cacheados: /home/palodo2/acl_classifier/data/slice_indices_final/val_coronal_indices.json


Some weights of ViTModel were not initialized from the model checkpoint at WinKawaks/vit-small-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  AUC coronal  (val): 0.9377
 Dataset cargado: 188 casos
  Plane: axial
  Augmentation: None (usando transform directo)
  Índices cacheados: /home/palodo2/acl_classifier/data/slice_indices_final/val_axial_indices.json


  AUC axial    (val): 0.9695

Umbral tau* = 0.3804  (n=188)
AUC 0.9837  Acc 0.9362  Prec 0.7500  Recall 1.0000  Esp 0.9211  F1 0.8571
VP 36  FN 0  FP 12  VN 140


## Test

Mismo umbral fijado en validación, aplicado al conjunto de test.

In [5]:
test_ens, test_lb = ensemble('test')
mt = calc(test_lb, test_ens, thr)
print(f'\nn={len(test_lb)}')
print(f"AUC {roc_auc_score(test_lb, test_ens):.4f}  Acc {mt['acc']:.4f}  Prec {mt['prec']:.4f}  "
      f"Recall {mt['rec']:.4f}  Esp {mt['spec']:.4f}  F1 {mt['f1']:.4f}")
print(f"VP {mt['tp']}  FN {mt['fn']}  FP {mt['fp']}  VN {mt['tn']}")

Some weights of ViTModel were not initialized from the model checkpoint at WinKawaks/vit-small-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Dataset cargado: 187 casos
  Plane: sagittal
  Augmentation: None (usando transform directo)
  Índices cacheados: /home/palodo2/acl_classifier/data/slice_indices_final/test_sagittal_indices.json


Some weights of ViTModel were not initialized from the model checkpoint at WinKawaks/vit-small-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  AUC sagittal (test): 0.8932
 Dataset cargado: 187 casos
  Plane: coronal
  Augmentation: None (usando transform directo)
  Índices cacheados: /home/palodo2/acl_classifier/data/slice_indices_final/test_coronal_indices.json


Some weights of ViTModel were not initialized from the model checkpoint at WinKawaks/vit-small-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  AUC coronal  (test): 0.8882
 Dataset cargado: 187 casos
  Plane: axial
  Augmentation: None (usando transform directo)
  Índices cacheados: /home/palodo2/acl_classifier/data/slice_indices_final/test_axial_indices.json


  AUC axial    (test): 0.9380

n=187
AUC 0.9433  Acc 0.8663  Prec 0.6552  Recall 0.8837  Esp 0.8611  F1 0.7525
VP 38  FN 5  FP 20  VN 124
